# Winosa ML — Subscriber Retention Analysis

> **Tujuan:** Memprediksi apakah subscriber akan tetap aktif (*retained*) atau berhenti (*unsubscribe*) berdasarkan data interaksi mereka dengan website Winosa.

---

**Cara pakai:** Klik **Run All** (atau jalankan cell satu per satu dengan `Shift+Enter`)

Notebook ini **self-contained** — generate data, EDA, training, dan simpan model semuanya ada di sini.

## 1. Import & Setup

In [ ]:
import os
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection    import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing      import StandardScaler, LabelEncoder
from sklearn.linear_model       import LogisticRegression
from sklearn.ensemble           import RandomForestClassifier
from sklearn.metrics            import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score, f1_score
)
from sklearn.pipeline           import Pipeline

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# ── Setup path (Windows-friendly) ─────────────────────────────────
# Notebook ada di ml/notebooks/, jadi naik 1 level ke ml/
BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR   = os.path.join(BASE_DIR, 'data')
MODELS_DIR = os.path.join(BASE_DIR, 'models')

os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print('✅ Libraries loaded')
print(f'   BASE_DIR   : {BASE_DIR}')
print(f'   DATA_DIR   : {DATA_DIR}')
print(f'   MODELS_DIR : {MODELS_DIR}')

## 2. Generate Dataset

Dataset dibuat secara **sintetis** karena data produksi Winosa belum tersedia. Disimulasikan 1.000 subscriber dengan perilaku engagement yang realistis.

In [ ]:
np.random.seed(42)
N = 1000

# ── Features ──────────────────────────────────────────────────────
days_subscribed   = np.random.randint(1, 730, N)
blog_views        = np.random.poisson(lam=days_subscribed / 60, size=N).clip(0, 50)
portfolio_clicks  = np.random.poisson(lam=2.5, size=N).clip(0, 20)
contact_submitted = np.random.binomial(1, 0.15, N)
service_views     = np.random.poisson(lam=1.8, size=N).clip(0, 10)
last_active_days  = np.random.exponential(scale=30, size=N).clip(0, 180).astype(int)
open_rate         = np.random.beta(2, 5, N).round(3)
plans             = np.random.choice(['free', 'basic', 'pro'], N, p=[0.6, 0.3, 0.1])

# ── Label: retained ───────────────────────────────────────────────
score = (
    0.3  * (blog_views / 50) +
    0.2  * (portfolio_clicks / 20) +
    0.15 * contact_submitted +
    0.15 * open_rate +
    0.1  * (service_views / 10) +
    -0.2 * (last_active_days / 180) +
    0.1  * np.where(plans == 'pro', 1, np.where(plans == 'basic', 0.5, 0))
)
prob_retained = 1 / (1 + np.exp(-5 * (score - 0.35)))
retained      = np.random.binomial(1, prob_retained, N)

df = pd.DataFrame({
    'days_subscribed':   days_subscribed,
    'blog_views':        blog_views,
    'portfolio_clicks':  portfolio_clicks,
    'contact_submitted': contact_submitted,
    'service_views':     service_views,
    'last_active_days':  last_active_days,
    'open_rate':         open_rate,
    'subscription_plan': plans,
    'retained':          retained,
})

CSV_PATH = os.path.join(DATA_DIR, 'subscribers.csv')
df.to_csv(CSV_PATH, index=False)

print(f'✅ Dataset saved → {CSV_PATH}')
print(f'   Shape    : {df.shape}')
print(f'   Retained : {retained.sum()} ({retained.mean()*100:.1f}%)')
print(f'   Unsub    : {(1-retained).sum()} ({(1-retained).mean()*100:.1f}%)')
df.head()

## 3. EDA — Distribusi & Target

In [ ]:
GOLD   = '#C4A832'
DARK   = '#1A1A1A'
ACCENT = '#E74C3C'

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Winosa — Subscriber EDA Overview', fontsize=15, fontweight='bold')

# 1. Class distribution
ax = axes[0, 0]
counts = df['retained'].value_counts()
bars = ax.bar(['Unsubscribed', 'Retained'], counts.values, color=[ACCENT, GOLD], edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Class Distribution', fontweight='bold')
ax.set_ylim(0, max(counts.values) * 1.2)

# 2. Blog views by retention
ax = axes[0, 1]
df.boxplot(column='blog_views', by='retained', ax=ax,
           boxprops=dict(color=DARK), medianprops=dict(color=GOLD, linewidth=2),
           whiskerprops=dict(color=DARK), capprops=dict(color=DARK))
ax.set_title('Blog Views by Retention', fontweight='bold')
ax.set_xlabel('Retained (0=No, 1=Yes)')
plt.sca(ax); plt.title('')

# 3. Open rate distribution
ax = axes[0, 2]
for val, label, color in [(0,'Unsubscribed',ACCENT),(1,'Retained',GOLD)]:
    ax.hist(df[df['retained']==val]['open_rate'], bins=25, alpha=0.7, color=color, label=label, edgecolor='white')
ax.set_title('Email Open Rate Distribution', fontweight='bold')
ax.set_xlabel('Open Rate')
ax.legend()

# 4. Days subscribed
ax = axes[1, 0]
for val, label, color in [(0,'Unsubscribed',ACCENT),(1,'Retained',GOLD)]:
    ax.hist(df[df['retained']==val]['days_subscribed'], bins=30, alpha=0.7, color=color, label=label, edgecolor='white')
ax.set_title('Days Subscribed Distribution', fontweight='bold')
ax.set_xlabel('Days')
ax.legend()

# 5. Retention by plan
ax = axes[1, 1]
plan_ret = df.groupby(['subscription_plan','retained']).size().unstack(fill_value=0)
plan_ret_pct = plan_ret.div(plan_ret.sum(axis=1), axis=0) * 100
plan_ret_pct.plot(kind='bar', ax=ax, color=[ACCENT, GOLD], edgecolor='white')
ax.set_title('Retention Rate by Plan', fontweight='bold')
ax.set_xlabel('Plan')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(['Unsubscribed','Retained'])

# 6. Last active days
ax = axes[1, 2]
df.boxplot(column='last_active_days', by='retained', ax=ax,
           boxprops=dict(color=DARK), medianprops=dict(color=GOLD, linewidth=2),
           whiskerprops=dict(color=DARK), capprops=dict(color=DARK))
ax.set_title('Days Since Last Active', fontweight='bold')
ax.set_xlabel('Retained (0=No, 1=Yes)')
plt.sca(ax); plt.title('')

plt.tight_layout()
overview_path = os.path.join(DATA_DIR, 'eda_overview.png')
plt.savefig(overview_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {overview_path}')

In [ ]:
# Correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Feature Correlation & Retention Rate by Quartile', fontsize=13, fontweight='bold')

numeric_cols = ['days_subscribed','blog_views','portfolio_clicks',
                'contact_submitted','service_views','last_active_days','open_rate','retained']
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='YlOrBr',
            ax=axes[0], linewidths=0.5, vmin=-1, vmax=1)
axes[0].set_title('Correlation Matrix', fontweight='bold')

ax = axes[1]
colors_list = [GOLD, '#2ECC71', '#3498DB', ACCENT]
for i, col in enumerate(['blog_views','open_rate','days_subscribed','last_active_days']):
    series = df.groupby(pd.qcut(df[col], q=4, duplicates='drop'))['retained'].mean() * 100
    ax.plot(range(len(series)), series.values, 'o-',
            color=colors_list[i], label=col.replace('_',' ').title(), linewidth=2, markersize=7)
ax.set_title('Retention Rate by Feature Quartile', fontweight='bold')
ax.set_xlabel('Quartile (low → high)')
ax.set_ylabel('Retention Rate (%)')
ax.legend(fontsize=9)
ax.set_xticks([0,1,2,3])
ax.set_xticklabels(['Q1','Q2','Q3','Q4'])

plt.tight_layout()
corr_path = os.path.join(DATA_DIR, 'eda_correlation.png')
plt.savefig(corr_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {corr_path}')

### Key Insights EDA

| Feature | Retained (mean) | Unsubscribed (mean) | Insight |
|---------|----------------|---------------------|---------|
| `blog_views` | lebih tinggi | lebih rendah | Engagement konten → retention |
| `last_active_days` | lebih rendah | lebih tinggi | Yang retained lebih sering aktif |
| `subscription_plan` | Pro: ~40% | Free: ~23% | Plan berbayar → retention lebih tinggi |
| `open_rate` | mirip | mirip | Fitur kurang diskriminatif sendirian |

> **Class imbalance:** ~74% unsubscribed, 26% retained → akan dihandle dengan `class_weight='balanced'`

## 4. Preprocessing

In [ ]:
# Load dataset
df = pd.read_csv(os.path.join(DATA_DIR, 'subscribers.csv'))

# Encode subscription_plan
le = LabelEncoder()
df['plan_encoded'] = le.fit_transform(df['subscription_plan'])
print('Label encoding subscription_plan:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

FEATURES = [
    'days_subscribed', 'blog_views', 'portfolio_clicks',
    'contact_submitted', 'service_views', 'last_active_days',
    'open_rate', 'plan_encoded'
]
TARGET = 'retained'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {len(X_train)} | Test: {len(X_test)}')
print(f'Train class balance: {y_train.value_counts().to_dict()}')

## 5. Model Training — LogisticRegression vs RandomForest

In [ ]:
models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=42, max_iter=1000,
                                   class_weight='balanced', C=1.0))
    ]),
    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42,
                                        class_weight='balanced', max_depth=10,
                                        min_samples_leaf=5))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print('=== 5-Fold Cross Validation (AUC-ROC) ===')
for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'  {name:22s}  AUC = {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
print('=== Test Set Evaluation ===')
eval_results = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    eval_results[name] = {
        'auc':      roc_auc_score(y_test, y_proba),
        'f1':       f1_score(y_test, y_pred),
        'acc':      accuracy_score(y_test, y_pred),
        'y_pred':   y_pred,
        'y_proba':  y_proba,
        'pipeline': pipeline,
    }
    r = eval_results[name]
    print(f'\n[{name}]')
    print(f'  AUC={r["auc"]:.4f}  F1={r["f1"]:.4f}  Acc={r["acc"]:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Unsubscribed','Retained']))

best_name  = max(eval_results, key=lambda k: eval_results[k]['auc'])
best_model = eval_results[best_name]['pipeline']
print(f'\n🏆 Best model: {best_name} (AUC={eval_results[best_name]["auc"]:.4f})')

## 6. Visualisasi Evaluasi

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Winosa ML — Model Evaluation Report (v1)', fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

GREEN = '#2ECC71'
RED   = '#E74C3C'

# ROC Curves
ax1 = fig.add_subplot(gs[0, 0])
for name, r in eval_results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    color = GOLD if name == best_name else '#95A5A6'
    ax1.plot(fpr, tpr, color=color, lw=2.5 if name==best_name else 1.5,
             label=f"{name} (AUC={r['auc']:.3f})")
ax1.plot([0,1],[0,1],'--', color='#BDC3C7', lw=1)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves', fontweight='bold')
ax1.legend(fontsize=8)

# Confusion Matrix
ax2 = fig.add_subplot(gs[0, 1])
cm = confusion_matrix(y_test, eval_results[best_name]['y_pred'])
im = ax2.imshow(cm, cmap='YlOrBr', aspect='auto')
for i in range(2):
    for j in range(2):
        ax2.text(j, i, str(cm[i,j]), ha='center', va='center',
                 fontsize=16, fontweight='bold',
                 color='white' if cm[i,j] > cm.max()/2 else DARK)
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(['Pred: Unsub','Pred: Retained'])
ax2.set_yticklabels(['Act: Unsub','Act: Retained'])
ax2.set_title(f'Confusion Matrix\n({best_name})', fontweight='bold')
plt.colorbar(im, ax=ax2, shrink=0.8)

# Metric Comparison
ax3 = fig.add_subplot(gs[0, 2])
metrics_keys = ['auc','f1','acc']
x = np.arange(len(metrics_keys))
width = 0.35
for i, (name, r) in enumerate(eval_results.items()):
    vals = [r[m] for m in metrics_keys]
    ax3.bar(x + (i-0.5)*width, vals, width,
            color=GOLD if name==best_name else '#95A5A6',
            label=name, edgecolor='white')
ax3.set_xticks(x)
ax3.set_xticklabels(['AUC-ROC','F1-Score','Accuracy'])
ax3.set_ylim(0, 1.1)
ax3.set_ylabel('Score')
ax3.set_title('Metric Comparison', fontweight='bold')
ax3.legend(fontsize=8)

# Feature Importance
ax4 = fig.add_subplot(gs[1, :2])
if best_name == 'RandomForest':
    clf = best_model.named_steps['clf']
    importances = clf.feature_importances_
    feat_names  = [f.replace('_',' ').title() for f in FEATURES]
    sorted_idx  = np.argsort(importances)
    bars = ax4.barh(np.array(feat_names)[sorted_idx], importances[sorted_idx],
                    color=GOLD, edgecolor='white')
    for i, bar in enumerate(bars):
        if i >= len(bars) - 3:
            bar.set_color(DARK)
    ax4.set_xlabel('Feature Importance')
    ax4.set_title(f'Feature Importance ({best_name})', fontweight='bold')
    for i, (idx, imp) in enumerate(zip(sorted_idx, importances[sorted_idx])):
        ax4.text(imp + 0.002, i, f'{imp:.3f}', va='center', fontsize=8)

# CV Comparison
ax5 = fig.add_subplot(gs[1, 2])
for name, scores in cv_results.items():
    color = GOLD if name==best_name else '#95A5A6'
    ax5.boxplot(scores, positions=[list(cv_results.keys()).index(name)],
                boxprops=dict(color=color), medianprops=dict(color=DARK, linewidth=2),
                whiskerprops=dict(color=color), capprops=dict(color=color))
ax5.set_xticks(range(len(cv_results)))
ax5.set_xticklabels(['LogReg','RandomForest'], fontsize=9)
ax5.set_ylabel('AUC-ROC')
ax5.set_title('CV AUC (5-fold)', fontweight='bold')
ax5.set_ylim(0.4, 1.0)

eval_path = os.path.join(DATA_DIR, 'model_evaluation.png')
plt.savefig(eval_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {eval_path}')

## 7. Simpan Model

In [ ]:
model_path    = os.path.join(MODELS_DIR, 'model_v1.pkl')
metadata_path = os.path.join(MODELS_DIR, 'model_v1_metadata.json')

metadata = {
    'model_name': best_name,
    'features':   FEATURES,
    'label_encoder_classes': le.classes_.tolist(),
    'metrics': {
        name: {
            'auc': round(float(r['auc']), 4),
            'f1':  round(float(r['f1']),  4),
            'acc': round(float(r['acc']), 4),
        }
        for name, r in eval_results.items()
    },
    'cv_results': {
        name: {'mean': round(float(s.mean()), 4), 'std': round(float(s.std()), 4)}
        for name, s in cv_results.items()
    },
}

joblib.dump({'pipeline': best_model, 'metadata': metadata, 'label_encoder': le}, model_path)
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✅ Model saved  → {model_path}')
print(f'✅ Metadata     → {metadata_path}')
print(f'\n=== SUMMARY ===')
print(f'  Best Model : {best_name}')
print(f'  AUC-ROC    : {eval_results[best_name]["auc"]:.4f}')
print(f'  F1-Score   : {eval_results[best_name]["f1"]:.4f}')
print(f'  Accuracy   : {eval_results[best_name]["acc"]:.4f}')

## 8. Test Inference (Demo Prediksi)

In [ ]:
# Load ulang model untuk verifikasi
saved    = joblib.load(model_path)
pipeline = saved['pipeline']

print(f'✅ Model loaded: {saved["metadata"]["model_name"]}')

# Contoh prediksi: subscriber aktif dengan plan basic
sample_aktif = pd.DataFrame([{
    'days_subscribed': 180, 'blog_views': 12, 'portfolio_clicks': 5,
    'contact_submitted': 1, 'service_views': 3, 'last_active_days': 3,
    'open_rate': 0.55, 'plan_encoded': 0  # basic=0
}])

# Contoh prediksi: subscriber tidak aktif dengan plan free
sample_pasif = pd.DataFrame([{
    'days_subscribed': 30, 'blog_views': 1, 'portfolio_clicks': 0,
    'contact_submitted': 0, 'service_views': 0, 'last_active_days': 90,
    'open_rate': 0.05, 'plan_encoded': 1  # free=1
}])

for label, sample in [('Subscriber AKTIF', sample_aktif), ('Subscriber PASIF', sample_pasif)]:
    pred = pipeline.predict_proba(sample)[0]
    status = '✅ RETAINED' if pred[1] > 0.5 else '❌ CHURN'
    print(f'\n{label}:')
    print(f'  P(unsubscribe) = {pred[0]:.3f}')
    print(f'  P(retained)    = {pred[1]:.3f}')
    print(f'  Prediksi       : {status}')

## 9. Kesimpulan & Next Steps

**Hasil Model v1:**
- Best model: **RandomForest** dengan AUC-ROC ~0.62
- Dataset menggunakan *synthetic data* sebagai baseline
- Class imbalance (74%/26%) ditangani dengan `class_weight='balanced'`
- AUC ~0.62 adalah **wajar untuk data sintetis** — akan meningkat signifikan dengan data produksi

**Next Steps (v2):**
1. Sambungkan ke data produksi via API Winosa backend
2. Feature engineering: RFM (Recency, Frequency, Monetary)
3. Hyperparameter tuning dengan `GridSearchCV` / `Optuna`
4. Coba `XGBoost` / `LightGBM`
5. SHAP values untuk explainability
6. Deploy model sebagai REST API endpoint